# Explore HS codes for UK critical and growth minerals

Scratch notebook: search the HMRC commodity lookup for each mineral and pick candidate codes.
Codes you settle on go into `config/minerals.toml`, with notes on anything ambiguous.

Run `python pipeline/fetch_commodity_lookup.py` first so the lookup file exists in `data/raw/`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

# Works whether the notebook runs from notebooks/ or from the repo root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Load the lookup saved by pipeline/fetch_commodity_lookup.py
lookup_file = REPO_ROOT / "data" / "raw" / "hmrc_commodity_lookup.json"
with open(lookup_file, encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

print(lookup_file.name, df.shape)
df.head()

In [ ]:
# Lower-level descriptions often omit the mineral name, so search all description levels together
TEXT_COLS = ["Hs4Description", "Hs6Description", "Cn8LongDescription"]
text = df[TEXT_COLS].fillna("").agg(" ".join, axis=1)

SHOW_COLS = ["CommodityId", "Cn8Code", "Hs6Code", "Hs4Description", "Hs6Description", "Cn8LongDescription"]

def search(term):
    """Return every lookup row whose descriptions mention the term (case-insensitive)."""
    return df.loc[text.str.contains(term, case=False), SHOW_COLS].sort_values("Cn8Code")

In [ ]:
search("cobalt")

In [ ]:
import tomllib

config = tomllib.load(open(REPO_ROOT / "config" / "minerals.toml", "rb"))["minerals"]
print(len(config), "minerals")
print(sum("critical" in m["lists"] for m in config.values()), "critical")
print(sum("growth" in m["lists"] for m in config.values()), "growth")

In [ ]:
missing = {k: sorted(set(m["cn8"]) - set(df["Cn8Code"])) for k, m in config.items()}
{k: v for k, v in missing.items() if v}